# TP 2 - Préparation améliorée de la base RAG

Ce notebook construit la base vectorielle V2 à partir des mêmes guides Markdown que la V1, avec un chunking structuré par en-têtes Markdown au lieu d'une fenêtre glissante par caractères.

### 0.1. Objectif
- **TP 2_1** : Préparer la base vectorielle V1 (chunking par caractères, embeddings, indexation Chroma)
- **TP 2_2** : Créer un assistant RAG simple
- **TP 2_3** : Préparer la base vectorielle V2 : le chunking suit la structure `###` des documents et ajoute un préfixe de contexte hiérarchique (`Document / Section / Sous-section`)
- **TP 2_4** : Créer un assistant RAG avec des méthodes avancées de retrieval

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
from shared.config import ROOT_DIR
from shared.rag_utils import (
    rag_load_markdown_documents,
    rag_chunk_markdown_document_by_headers,
    rag_describe_chunks,
    rag_index_chunks_chroma,
    # INFO : Choix entre local ou cloud (embeddings)
    rag_embed_all_chunks, # Cloud
    #rag_embed_all_chunks_local as rag_embed_all_chunks, # Local
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
MARKDOWN_DIR = DATA_DIR / "guides_markdown"
CHROMA_DIR_V2 = DATA_DIR / "chroma_db_rag_v2"

MIN_CHUNK_CHARS = 250
MAX_CHUNK_CHARS = 3000
EMBEDDING_BATCH_SIZE = 16

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `rag_load_markdown_documents` : charge les fichiers `.md` d'un dossier en `MarkdownDocument`
- `rag_chunk_document_by_chars` : fonction qui découpe un document en chunks de taille fixe avec fenêtre glissante
- `rag_chunk_markdown_document_by_headers` : fonction qui découpe un document selon la structure Markdown avec contexte hiérarchique
- `rag_describe_chunks` : fonction qui affiche des statistiques et la distribution des tailles de chunks
- `rag_embed_text_batch` : fonction qui calcule les embeddings d'une liste de textes via l'API Google GenAI
- `rag_embed_all_chunks` : fonction qui calcule les embeddings de tous les chunks par batch
- `rag_index_chunks_chroma` : fonction qui indexe des chunks vectorisés dans une collection Chroma

--> Disponibles dans `shared/rag_utils.py`

### 0.4. Charger les documents

Les guides ont déjà été convertis de PDF vers Markdown et nettoyés.
Concrètement, le Markdown rend le découpage plus fiable : **titres explicites** (`#`, `##`, `###`), retours ligne propres, et moins d'artefacts parasites.
C'est cette structure qui permet le **découpage par en-têtes**.

In [ ]:
documents = rag_load_markdown_documents(MARKDOWN_DIR)

total_characters = sum(len(doc["text"]) for doc in documents)
print(f"Documents Markdown chargés : {len(documents)}")
print(f"Nombre total de caractères : {total_characters}\n")
for doc in documents:
    print(f"  {doc['source']}: {len(doc['text'])} caractères")

### 0.5. Inspecter la structure Markdown

Vérifier la **hiérarchie des titres** d'un document.
Le chunker découpe sur `###` et remonte les parents `#`/`##` pour construire le **préfixe de contexte**.

In [ ]:
sample_doc = documents[0]
print(f"Document : {sample_doc['source']}")
print(f"Total : {len(sample_doc['text'])} caractères\n")
print(sample_doc["text"][:800])

---
## 1. Stratégie de chunking V2 : basée sur les en-têtes Markdown

La V1 découpe par **fenêtre glissante** de taille fixe, sans tenir compte de la structure du document.

La V2 s'appuie sur les **en-têtes Markdown** (`#`, `##`, `###`) pour découper à des frontières sémantiques naturelles. Chaque chunk commence par un **préfixe de contexte hiérarchique** construit à partir des titres parents, ce qui améliore la qualité de la recherche vectorielle.

### 1.1. Appliquer le chunking à tous les documents

In [ ]:
chunks = []
for doc in documents:
    chunks.extend(rag_chunk_markdown_document_by_headers(doc, min_chunk_chars=MIN_CHUNK_CHARS, max_chunk_chars=MAX_CHUNK_CHARS))

### 1.2. Inspecter les chunks (statistiques)

In [ ]:
rag_describe_chunks(chunks)

### 1.3. Inspecter les chunks avec préfixe de contexte

Chaque chunk doit commencer par un **en-tête de contexte** (`Document / Section / Sous-section / Paragraphe`) avant le contenu.
Comparer visuellement avec un chunk V1 (`2_1`) pour vérifier le gain de contexte.

In [ ]:
sample_indices = [0, len(chunks) // 6, len(chunks) // 5, len(chunks) // 4, len(chunks) // 2, 3 * len(chunks) // 4, len(chunks) - 1]
for idx in sample_indices:
    chunk = chunks[idx]
    print(f"--- Chunk #{chunk.chunk_id} | source: {chunk.source} | {chunk.char_count} caractères ---")
    print(chunk.text)
    print("\n"*5)

---
## 2. Embeddings et indexation

Même principe qu'en 2_1 (voir la partie Embeddings et indexation) : `rag_embed_all_chunks` calcule les embeddings par lots, puis `rag_index_chunks_chroma` les indexe dans Chroma. Les deux fonctions sont déjà codées, on les réutilise ici sur les chunks V2.

### 2.1. Calculer les embeddings des chunks

In [ ]:
chunk_embeddings = rag_embed_all_chunks(chunks=chunks, batch_size=EMBEDDING_BATCH_SIZE)

### 2.2. Indexer les chunks dans Chroma

In [ ]:
rag_index_chunks_chroma(persist_dir=CHROMA_DIR_V2, chunks=chunk_embeddings)

print(f"V2 - Chunks indexés : {len(chunk_embeddings)}")
print(f"V2 - Dimension des embeddings : {len(chunk_embeddings[0].embedding)}")